# 07. HybridScoreFollower: HSMM + OLTW + anchor recovery

Это **боевой трекер** проекта (используется в `interactive_tester.py` и `real_orchestra_player.py`). Идея:

1. Параллельно работают **два независимых** трекера — `ScoreFollowerHSMM` и `ScoreFollowerOLTW` — и каждый видит каждое событие.
2. На каждом шаге смотрим **confidence** HSMM (это `max(α)`):
   - confidence высокая → берём ответ HSMM,
   - confidence низкая → берём ответ OLTW (он жёсткий, всегда отвечает),
3. Параллельно работает **anchor-search**: последние K событий ищутся как окно в score MIDI с помощью template-matching по pitch + temp-fitting. Если нашлось хорошее совпадение далеко от текущей позиции — это **resync**, оба трекера принудительно перепрыгивают.

Все три механизма (HSMM + OLTW + якорь) **дополняют** друг друга. Это и есть «классический вероятностный трекер» из тезисов; RL-агент тезисов будет работать поверх него.

В этом ноутбуке мы посмотрим, **когда** активируется каждый механизм.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt

from musictech.core import HybridScoreFollower
from dataset_viewer import load_performance

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
DATA = PROJECT_ROOT / "generated_dataset"

In [ ]:
def run_hybrid(score_path: Path, perf_path: Path):
    follower = HybridScoreFollower(str(score_path), load_tuning_profile=False)
    performance = load_performance(perf_path)
    rows = []
    for event in performance:
        idx = follower.process_event(event["pitch"], event["timestamp"])
        rows.append({
            "event_pitch": event["pitch"],
            "event_time": event["timestamp"],
            "current_index": idx,
            "hsmm_index": follower.last_hsmm_index,
            "oltw_index": follower.last_oltw_index,
            "selected_model": follower.last_selected_model,
            "confidence": follower.confidence,
            "resynced": follower.last_resynced,
            "anchor_target": follower.last_anchor_target,
        })
    return follower, rows

## 1. Сценарий `noisy`: где смотрим разницу между подсистемами

На синтетике `noisy` пианист (синтетически) делает опечатки. HSMM теряет уверенность, OLTW спасает. Сравним их выводы по шагам.

In [ ]:
follower, rows = run_hybrid(DATA / "noisy.json", DATA / "noisy.mid")

import pandas as pd
df = pd.DataFrame(rows)
df[["event_pitch", "event_time", "current_index", "hsmm_index", "oltw_index", "selected_model", "confidence", "resynced"]].round(3)

## 2. График: где работает HSMM, где OLTW

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
events = np.arange(len(df))

axes[0].plot(events, df["current_index"], marker="o", label="current (chosen)", color="black")
axes[0].plot(events, df["hsmm_index"], marker="s", linestyle="--", alpha=0.5, label="HSMM")
axes[0].plot(events, df["oltw_index"], marker="x", linestyle="--", alpha=0.5, label="OLTW")
axes[0].set_ylabel("score state index")
axes[0].set_title("Hybrid: outputs of each sub-tracker on 'noisy'")
axes[0].legend()

axes[1].plot(events, df["confidence"], marker="o", color="teal")
axes[1].axhline(follower.confidence_threshold, color="red", linestyle=":",
               label=f"threshold = {follower.confidence_threshold}")
axes[1].set_xlabel("event #")
axes[1].set_ylabel("HSMM confidence = max(α)")
axes[1].set_title("Confidence of HSMM — when it drops, OLTW takes over")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Anchor recovery

На синтетике 13 событий — anchor-window не успевает прогреться (минимальное окно `anchor_window_lengths[-1]` = 4). На реальном `rach_solo` якорь активно срабатывает при сильных скачках. Здесь покажем только что **API доступен**:

In [ ]:
print("anchor_window_lengths     :", follower.anchor_window_lengths)
print("last_anchor_target        :", follower.last_anchor_target)
print("last_anchor_cost          :", follower.last_anchor_cost)
print("last_anchor_window        :", follower.last_anchor_window)
print()
print("anchor events (pitches)   :", follower.anchor_event_pitches[:10], "...")
print("anchor score positions    :", follower.anchor_event_score_positions[:10], "...")

## 4. Атрибуты гибрида, которые нужны RL-агенту

Согласно тезисам, `s_t` = (α̂_t, история темпа, история эмиссионных ошибок, относительная позиция). Что нам уже доступно из `HybridScoreFollower`:

| В тезисах | В коде |
|---|---|
| `α̂_t` (сжатая) | `follower.hsmm.alpha` → собираем `AlphaSummary` (см. ноутбук 09) |
| `current_index` | `follower.current_index` |
| `N` | `follower.N` |
| `model_label` | `follower.last_selected_model` или `follower.mode_label` |
| `confidence` | `follower.confidence` |
| `e_t` (эмиссионная ошибка) | `follower.hsmm.last_best_pitch_distance` |
| `τ_t` (темп) | будет в `TempoTracker` — см. ноутбук 08 |